# ML-08 — Capstone Modeling Lane
Refresh / Content Opportunity Scoring

Working with an AI assistant: read `skills/README.md`, then load `training-honest-models` + `flyrank/flyrank-data` — done before writing this notebook.


## 1. Method choice and why

**Target:** I'm reusing the exact proxy target from my own ML-03 framing (already in my portfolio), not inventing a new one: `is_opportunity = (trend_direction == "down") AND (avg_position > 10)` — a yes/no label with an observed value in the data.

**Method:** Per the `training-honest-models` skill's table, a yes/no question with an observed label starts with **Logistic Regression** (readable — I can see and explain every coefficient), then **Random Forest** (stronger, if it earns its complexity). I'm not reaching for Gradient Boosting here: the skill is explicit that added complexity has to earn its place over a model I can print and read, and I want to see whether the readable model is already good enough before justifying the more opaque one.

**Why this fits my lane:** the Week-4 baseline was a hand-written CTR-gap rule, not a rule aimed at this exact label. Comparing a *trained* model against that rule, on the same label and the same held-out data, tells me directly whether learning from the data actually beats my own hand-written logic — which is the whole point of this week.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same proxy label as my ML-03 framing doc.
df["is_opportunity"] = ((df["trend_direction"] == "down") & (df["avg_position"] > 10)).astype(int)
print("Base rate of is_opportunity:", round(df["is_opportunity"].mean(), 3), "| n =", len(df))


Base rate of is_opportunity: 0.298 | n = 30000


## 2. Split design

**Grouped by client, not random, and not time-aware** (this dataset doesn't carry a real per-row date column — it's already aggregated into 90-day/30-day windows) — so a **grouped split by `client_id`** is the honest choice: no client's pages appear in both train and validation, which prevents the model from just memorizing one client's typical traffic pattern and getting credit for it on that same client's held-out pages. A plain random split would let rows from the same client leak across both sides, inflating the score without teaching the model anything that generalizes to a genuinely new client.


In [2]:
# --- Recreate the Week-4 baseline score exactly, so it's evaluated on the same data ---
tier_stats = df.groupby("position_tier").agg(
    total_clicks=("clicks_90d", "sum"), total_impr=("impressions_90d", "sum")
)
tier_stats["expected_ctr_pct"] = tier_stats["total_clicks"] / tier_stats["total_impr"] * 100
df["expected_ctr_pct"] = df["position_tier"].map(tier_stats["expected_ctr_pct"].to_dict())
df["ctr_gap"] = df["expected_ctr_pct"] - df["ctr"]
df["visible"] = (df["impressions_90d"] >= 300).astype(int)
df["ctr_gap_positive"] = df["ctr_gap"].clip(lower=0)
df["baseline_score"] = df["visible"] * df["ctr_gap_positive"] * df["impressions_90d"]

# --- Grouped split by client_id ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(df, groups=df["client_id"]))
train, val = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()

print("Train rows:", len(train), "| Val rows:", len(val))
print("Train clients:", train["client_id"].nunique(), "| Val clients:", val["client_id"].nunique())
print("Client overlap between train/val:", len(set(train["client_id"]) & set(val["client_id"])), "(must be 0)")


Train rows: 22885 | Val rows: 7115
Train clients: 24 | Val clients: 8
Client overlap between train/val: 0 (must be 0)


## 3. Train + compare vs my baseline

**Leakage catch, kept in on purpose:** my first pass at features included `impressions_last_30d` and `impressions_prev_30d`. Both looked like reasonable trend signals — until I checked the data dictionary and confirmed `trend_direction` (my label's source) is **literally computed** from those two columns (last-30d vs prev-30d impressions, ±20% thresholds). Using them as features would have let the model see most of the answer directly, not learn from it. I dropped both. `trend_pct` and `trend_direction` themselves were already excluded as the obvious label source; `avg_position` and `position_tier` are excluded too, since `avg_position > 10` is literally half of my label's definition. The numbers below are **after** removing all four.


In [3]:
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "sessions_last_30d",      # kept: not part of the trend_direction formula
    "clicks_prev_30d", "sessions_prev_30d",      # kept: not part of the trend_direction formula
    "content_age_days", "days_since_last_update",
    "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
categorical_features = ["content_type", "main_intent", "competition_level", "freshness_tier",
                         "provider_used", "model_used"]

for c in categorical_features:
    df[c] = df[c].fillna("missing")
train, val = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()

X_train, y_train = train[numeric_features + categorical_features], train["is_opportunity"]
X_val, y_val = val[numeric_features + categorical_features], val["is_opportunity"]

preprocess_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])
preprocess_lr = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

lr_pipe = Pipeline([("prep", preprocess_lr), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))])
lr_pipe.fit(X_train, y_train)
lr_probs = lr_pipe.predict_proba(X_val)[:, 1]

rf_pipe = Pipeline([("prep", preprocess_rf), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_SEED, n_jobs=-1))])
rf_pipe.fit(X_train, y_train)
rf_probs = rf_pipe.predict_proba(X_val)[:, 1]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

print(f"{'model':35s}  ROC-AUC   Precision@50   Precision@20")
for name, scores in [("baseline_score (Week-4 rule)", val["baseline_score"].values),
                      ("logistic_regression", lr_probs),
                      ("random_forest", rf_probs)]:
    auc = roc_auc_score(y_val, scores)
    p50 = precision_at_k(y_val, scores, 50)
    p20 = precision_at_k(y_val, scores, 20)
    print(f"{name:35s}  {auc:.3f}     {p50:.3f}          {p20:.3f}")

print("\nBase rate of is_opportunity in validation set:", round(y_val.mean(), 3))


model                                ROC-AUC   Precision@50   Precision@20
baseline_score (Week-4 rule)         0.536     0.040          0.000
logistic_regression                  0.600     0.260          0.350
random_forest                        0.573     0.320          0.350

Base rate of is_opportunity in validation set: 0.232


**Comparison table (val set, n = 7,115, base rate = 0.232):**

| Model | ROC-AUC | Precision@50 | Precision@20 |
|---|---|---|---|
| Baseline score (Week-4 rule) | 0.536 | 0.040 | 0.000 |
| Logistic Regression | 0.600 | 0.260 | 0.350 |
| Random Forest | 0.573 | 0.320 | 0.350 |

**Reading this honestly:** both trained models clearly beat the Week-4 baseline — the baseline's Precision@50 (0.040) is actually *worse* than picking pages at random (base rate 0.232), which makes sense once I think about it: the baseline rule was built to find CTR-fix opportunities specifically, not "declining and poorly positioned" pages — it was never aimed at this label, so losing to random selection on this exact target isn't a surprise, it's a reminder that a rule is only as good as the question it was built to answer.

Between the two trained models, **Random Forest wins on Precision@50 (0.320 vs 0.260) despite a lower ROC-AUC (0.573 vs 0.600)** than Logistic Regression. Per the skill's own guidance — "if the model wins at precision@50 but loses at precision@20... report both; that IS the finding" — I'm reporting both rather than picking whichever number favors one model. AUC measures ranking quality across the *whole* dataset; Precision@50 only cares about the very top of the list, which is what actually matters for a content team working through a prioritized queue. Random Forest's ability to catch nonlinear patterns (e.g., an interaction between low engagement AND long content age) seems to help right at the top of the ranking, even though it's slightly noisier overall.


## 4. Errors and interpretation

**What the model leans on (permutation importance, Random Forest):**


In [4]:
perm = permutation_importance(rf_pipe, X_val, y_val, n_repeats=8, random_state=RANDOM_SEED, scoring="roc_auc", n_jobs=-1)
feat_names = numeric_features + categorical_features
imp_df = pd.DataFrame({"feature": feat_names, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
print(imp_df.head(10).to_string(index=False))


              feature  importance
days_with_impressions    0.028892
      clicks_last_30d    0.016371
                  ctr    0.011750
      impressions_90d    0.009993
           clicks_90d    0.008509
      clicks_prev_30d    0.005010
         content_type    0.003016
           model_used    0.002668
        provider_used    0.002194
          competition    0.001830


**Top feature: `days_with_impressions`.** This makes sense rather than looking suspiciously perfect — a page that only shows up in search results on a handful of days over 90 days is inherently a weaker, more volatile signal than one visible almost every day, and volatility is exactly the kind of thing that would make a page look like it's "declining" in a proxy label built from a threshold comparison. `clicks_last_30d` and `ctr` follow, which are genuine engagement signals, not leakage — they weren't used anywhere in constructing the label itself.


In [5]:
val = val.reset_index(drop=True)
val["rf_prob"] = rf_probs
val["rf_pred"] = (val["rf_prob"] >= 0.5).astype(int)
wrong = val[val["rf_pred"] != val["is_opportunity"]].copy()

false_pos = wrong[wrong["rf_pred"] == 1].sort_values("rf_prob", ascending=False).head(3)
false_neg = wrong[wrong["rf_pred"] == 0].sort_values("rf_prob").head(3)

print("=== 3 false positives (model said 'opportunity', wasn't) ===")
print(false_pos[["content_id", "ctr", "impressions_90d", "freshness_tier", "rf_prob"]].to_string(index=False))
print()
print("=== 3 false negatives (model missed a real opportunity) ===")
print(false_neg[["content_id", "ctr", "impressions_90d", "freshness_tier", "rf_prob"]].to_string(index=False))


=== 3 false positives (model said 'opportunity', wasn't) ===
          content_id  ctr  impressions_90d freshness_tier  rf_prob
content_958a46db26bd  0.0              198           181+ 0.642567
content_d34c89fad803  0.0               80           181+ 0.641673
content_30eb41dff556  0.0               84           181+ 0.641308

=== 3 false negatives (model missed a real opportunity) ===
          content_id  ctr  impressions_90d freshness_tier  rf_prob
content_8d65431b49ac 0.35            56302           0-30 0.082177
content_e9bc92689a75 0.00                1           0-30 0.084012
content_16f38acf0f26 0.00                2           0-30 0.086127


**Why these are hard:**

- **False positives** are all very stale (`181+` freshness), zero-CTR, but genuinely *low-impression* pages (80–200 impressions over 90 days). The model reads "stale + zero clicks" as a strong decline signal, but at this little volume there may simply not be enough traffic for `trend_direction` to register as anything other than flat or stable — the model is pattern-matching to a signal (staleness) that's usually predictive but breaks down at very low visibility, where there isn't enough traffic for a real trend to show up either way.
- **False negatives** are the opposite problem: freshly updated (`0-30` freshness) pages that are still genuinely declining. The model appears to treat recent freshness as reassuring, which is usually reasonable, but it means a page that was updated recently and is *still* underperforming gets under-flagged — freshness is being used as a stand-in for "healthy," and that assumption breaks for content that was weak from the start rather than stale.

**Does this reward complexity alone?** No — Logistic Regression is competitive with Random Forest on AUC (0.600 vs 0.573) and only loses on Precision@50. Given how close they are, and that the readable model explains itself directly through coefficients, I'd deploy Logistic Regression first and only justify Random Forest's opacity if Precision@50 turns out to matter more than overall ranking quality for how a content team will actually use this list.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are the repo's own pseudonyms)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.
